In [ ]:
#STAGE 2 - COGNITIVE REGRESSION (GROUP-STRATIFIED)

"""
Tests the association between cognitive z-scores and confounder-adjusted
biomarker levels (from Stage 1). Restricts the analysis to one of two
positively-defined cohorts:
- 'hc': observations flagged hc == 1
- 'ic': observations flagged ic == 1
Rows belonging to neither group are excluded.
Manually change the GROUP value to run on each group. 'all' indicates the entire cohort
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import os

GROUP = 'all'  # 'hc', 'all' or 'ic'

output_dir = f'2stage_{GROUP}/'
os.makedirs(output_dir, exist_ok=True)

data_0 = pd.read_csv('stage1_adjusted_data.csv')

def filter_group(df, group):
    if group not in ('hc', 'ic', 'all'):
        raise ValueError(f"GROUP must be 'hc', 'ic', or 'all', got {group!r}")
    if group == 'all':
        out = df.copy()
        print(f"Group 'all': {out['eid'].nunique()} people, {len(out)} observations (no filter)")
    else:
        out = df[df[group] == 1].copy()
        print(f"Group '{group}': {out['eid'].nunique()} people, {len(out)} observations")
    return out
    
data_0 = filter_group(data_0, GROUP)
biomarkers = ['gfap', 'nefl']
tests = data_0['test_name'].unique()

In [ ]:
#STAGE 2 - ANALYZE ADJUSTED BIOMARKERS

"""
Simple regression of adjusted biomarker on z_adj, for each (biomarker, test)
combination within the selected group. Confounders are already removed in Stage 1.
"""

lm_all_results = []

def format_pvalue(p):
    return f"{p:.2e}" if p < 0.001 else f"{p:.4f}"

for biomarker in biomarkers:
    adjusted_biomarker = f'{biomarker}_adjusted'
    print(f"\nAnalysing adjusted {biomarker.upper()}")

    for test in tests:
        test_df = data_0[data_0['test_name'] == test].copy()
        test_df = test_df[['eid', 'z_adj', adjusted_biomarker, 'test_name']].dropna()

        # Skip tests with insufficient data within this group
        if len(test_df) < 30:
            print(f"  SKIPPED {test}: only {len(test_df)} observations in {GROUP} group")
            continue

        model = smf.ols(f'{adjusted_biomarker} ~ z_adj', data=test_df).fit()

        zscore_estimate = model.params['z_adj']
        zscore_se = model.bse['z_adj']
        zscore_pvalue = model.pvalues['z_adj']
        zscore_ci = model.conf_int().loc['z_adj']

        result = {
            'group': GROUP,
            'biomarker': biomarker,
            'test': test,
            'n': len(test_df),
            'estimate': zscore_estimate,
            'std_error': zscore_se,
            'ci_low': zscore_ci[0],
            'ci_high': zscore_ci[1],
            't_statistic': model.tvalues['z_adj'],
            'p_value': zscore_pvalue,
            'r_squared': model.rsquared,
            'adj_r_squared': model.rsquared_adj
        }
        lm_all_results.append(result)

        print(f"  {test}: β={zscore_estimate:.4f}, "
              f"95%CI [{zscore_ci[0]:.4f}, {zscore_ci[1]:.4f}], "
              f"p={format_pvalue(zscore_pvalue)}, n={len(test_df)}")

        # Per-test output files
        test_dir = os.path.join(output_dir, 'adjusted_analysis', biomarker, test)
        os.makedirs(test_dir, exist_ok=True)

        coef_table = pd.DataFrame({
            'term': model.params.index,
            'estimate': model.params.values,
            'std_error': model.bse.values,
            't_statistic': model.tvalues.values,
            'p_value': model.pvalues.values,
            'ci_low': model.conf_int()[0].values,
            'ci_high': model.conf_int()[1].values
        })
        coef_table.to_csv(os.path.join(test_dir, 'coefficients.csv'), index=False)

print(f"\nAll tests complete for group: {GROUP}")

In [ ]:
#CREATE RESULTS SUMMARY TABLE
"""
This block compiles the necessary results for reporting 
It takes lm_all_results as the main input, defines it as a dataframe and organises the output neatly
"""
#Organise, view and save the raw results
all_results = pd.DataFrame(lm_all_results)
all_results.to_csv(os.path.join(output_dir, 'all_results_raw.csv'), index=False)
print("Raw results:\n")
print(all_results.to_string(index=False))

#Generate formated table

# Function to add significance stars
def add_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

# Function to format p-values for table
def format_pvalue_table(p):
    if p < 0.001:
        return f"{p:.2e}"   # Scientific notation
    elif p < 0.01:
        return f"{p:.4f}"   # 4 decimal places
    else:
        return f"{p:.3f}"   # 3 decimal place       

pub_table = all_results.copy()
pub_table['Biomarker'] = pub_table['biomarker'].str.upper()
pub_table['Test'] = pub_table['test']
pub_table['N'] = pub_table['n']
pub_table['β'] = pub_table['estimate'].round(4)
pub_table['SE'] = pub_table['std_error'].round(4)
pub_table['95% CI'] = pub_table.apply(lambda row: f"[{row['ci_low']:.4f}, {row['ci_high']:.4f}]", axis=1)
pub_table['t'] = pub_table['t_statistic'].round(3)
pub_table['R²'] = pub_table['r_squared'].apply(lambda x: f"{x:.5f}")
pub_table['p'] = pub_table['p_value'].apply(format_pvalue_table)
pub_table[''] = pub_table['p_value'].apply(add_stars)

# Select final columns
pub_table = pub_table[['Biomarker', 'Test', 'N', 'β', 'SE', '95% CI', 't', 'R²', 'p', '']]

# Save as CSV
pub_table.to_csv(os.path.join(output_dir, 'publication_table.csv'), index=False)

print("\nFormatted results:\n")
print(pub_table.to_string(index=False))

In [ ]:
#PERCENTAGE CHANGE IN RELATIVE PROTEIN EXPRESSION
"""
Converts raw beta (in NPX log2 units) to % change in relative protein expression.
NPX differences preserve fold-change: Δ1 NPX = 2-fold change.
Formula: % change = (2^(-β) - 1) * 100
 
Sign is negated so positive % = increase with WORSE cognition.
CIs swap when negating.
"""

# Load raw results
all_results = pd.read_csv(os.path.join(output_dir, 'all_results_raw.csv'))
 
# Compute percentage change directly from raw beta
all_results['pct_change'] = (2**(-all_results['estimate']) - 1) * 100
all_results['pct_ci_low'] = (2**(-all_results['ci_high']) - 1) * 100
all_results['pct_ci_high'] = (2**(-all_results['ci_low']) - 1) * 100
 
# Update raw results CSV
all_results.to_csv(os.path.join(output_dir, 'all_results_raw.csv'), index=False)
 
# Update publication table
pub_table = pd.read_csv(os.path.join(output_dir, 'publication_table.csv'))
pub_table['% Δ'] = all_results['pct_change'].apply(lambda x: f"{x:+.2f}%")
pub_table['% Δ 95% CI'] = all_results.apply(
    lambda row: f"[{row['pct_ci_low']:+.2f}, {row['pct_ci_high']:+.2f}]", axis=1)
pub_table.to_csv(os.path.join(output_dir, 'publication_table.csv'), index=False)
 
print("Percentage change in relative protein expression per 1 SD worse cognition")
print("=" * 85)
print(pub_table[['Biomarker', 'Test', 'β', '% Δ', '% Δ 95% CI', 'p',]].to_string(index=False))

In [ ]:
# FOREST PLOT — Percentage change in relative protein expression
"""
This is the most clinically interpretable plot.
X-axis shows % change in relative protein expression per 1 SD worse cognition.
Derived directly from raw beta: (2^(-β) - 1) * 100
"""
 
# Load results (must have pct_change columns from previous cell)
all_results = pd.read_csv(os.path.join(output_dir, 'all_results_raw.csv'))
 
def format_pvalue(p):
    if p < 0.001:
        return 'p<0.001'
    elif p < 0.01:
        return f'p={p:.3f}'
    else:
        return f'p={p:.2f}'
 
for biomarker in biomarkers:
    plot_data = all_results[all_results['biomarker'] == biomarker].copy()
    plot_data = plot_data.sort_values('pct_change')
    n_tests = len(plot_data)
 
    fig, ax = plt.subplots(figsize=(10, max(4, n_tests * 0.4)))
    y_pos = np.arange(n_tests)
 
    ax.scatter(plot_data['pct_change'], y_pos, s=100, color='black', zorder=3,
               edgecolors='black', linewidths=1.5)
 
    for i, (idx, row) in enumerate(plot_data.iterrows()):
        ax.plot([row['pct_ci_low'], row['pct_ci_high']], [i, i],
                'k-', linewidth=2, zorder=2)
 
    ax.axvline(0, color='darkgrey', linestyle='--', linewidth=1, alpha=0.8)
 
    #Add % value and p-value labels
    for i, (idx, row) in enumerate(plot_data.iterrows()):
        p_text = format_pvalue(row['p_value'])
        x_pos = plot_data['pct_ci_high'].max() + 0.15
        ax.text(x_pos, i, f"{p_text}",
                fontsize=12, va='center', ha='left')
 
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_data['test'], fontsize=16)
    ax.tick_params(axis='x', labelsize=16)
    ax.set_xlabel('Percentage change', 
                  fontsize=18, fontweight='bold')
    ax.set_ylabel('Cognitive Test', fontsize=18, fontweight='bold')
    ax.set_title(f'{biomarker.upper()}: Protein Expression per SD Cognitive Impairment', 
                 fontsize=20, fontweight='bold')
    ax.grid(axis='x', alpha=0.3, linewidth=1)
 
    ax.spines['top'].set_linewidth(1)
    ax.spines['right'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)
 
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{biomarker}_forest_pct_change.svg'), 
                dpi=800, bbox_inches='tight', facecolor='white', format='svg')
    plt.show()
 
print("✓ Percentage change forest plots created")

In [ ]:
#STAGE 2 - COMBINED COGNITIVE TESTS (COMPOSITE SCORE)

"""
Builds a single global-cognition composite per person-instance by averaging
the directionally-aligned z-scores across the four baseline tests
(FI, NM, PMat, RT), then runs the SAME model as the per-test cell:
adjusted_biomarker ~ z_composite.

This is the correct pooled analogue: one biomarker value matched to one
cognitive score per row. (Stacking all tests with test fixed effects collapses
the slope to ~0, because the biomarker does not vary by test within a sitting.)

Direction: RT and PMat are deficit tests (higher = worse); their z is flipped
so the composite is oriented higher = better, matching FI/NM. The composite
averages whatever of the four tests a person-instance has available.
"""

COMPOSITE_TESTS = ['FI', 'NM', 'PMat', 'RT']
DEFICIT_TESTS = ['RT', 'PMat']   # higher = worse; flip sign

def format_pvalue(p):
    return f"{p:.2e}" if p < 0.001 else f"{p:.4f}"

dat = data_0[data_0['test_name'].isin(COMPOSITE_TESTS)].copy()
dat['z_dir'] = np.where(dat['test_name'].isin(DEFICIT_TESTS), -dat['z_adj'], dat['z_adj'])

inst_cols = [c for c in ['eid', 'instance'] if c in dat.columns]
group_cols = inst_cols if len(inst_cols) > 1 else ['eid']

combined_results = []
for biomarker in biomarkers:
    adjusted_biomarker = f'{biomarker}_adjusted'
    print(f"\nCombined composite ({'+'.join(COMPOSITE_TESTS)}) - adjusted {biomarker.upper()}")

    sub = dat[group_cols + ['z_dir', adjusted_biomarker]].dropna()

    # average available tests to one score per person-instance
    comp = (sub.groupby(group_cols)
               .agg(z_composite=('z_dir', 'mean'),
                    **{adjusted_biomarker: (adjusted_biomarker, 'first')})
               .reset_index())

    if len(comp) < 30:
        print(f"  SKIPPED: only {len(comp)} person-instances")
        continue

    model = smf.ols(f'{adjusted_biomarker} ~ z_composite', data=comp).fit()
    est = -model.params['z_composite']
    se = model.bse['z_composite']
    p = model.pvalues['z_composite']
    ci_raw = model.conf_int().loc['z_composite']
    ci = [-ci_raw[1], -ci_raw[0]]

    result = {
        'group': GROUP, 'biomarker': biomarker, 'test': 'COMBINED',
        'n': len(comp), 'n_people': comp['eid'].nunique(),
        'estimate': est, 'std_error': se,
        'ci_low': ci[0], 'ci_high': ci[1],
        't_statistic': model.tvalues['z_composite'], 'p_value': p,
        'r_squared': model.rsquared, 'adj_r_squared': model.rsquared_adj,
    }
    combined_results.append(result)

    print(f"  β={est:.4f}, 95%CI [{ci[0]:.4f}, {ci[1]:.4f}], "
          f"p={format_pvalue(p)}, n={len(comp)} ({comp['eid'].nunique()} people)")

    comb_dir = os.path.join(output_dir, 'adjusted_analysis', biomarker, 'COMBINED')
    os.makedirs(comb_dir, exist_ok=True)
    pd.DataFrame([result]).to_csv(os.path.join(comb_dir, 'coefficients.csv'), index=False)

combined_df = pd.DataFrame(combined_results)
combined_df.to_csv(os.path.join(output_dir, 'combined_results_raw.csv'), index=False)
print(f"\nCombined composite analysis complete for group: {GROUP}")

In [ ]:
#COMBINED TESTS - PERCENTAGE CHANGE + APPEND TO MAIN RESULTS
"""
Converts the pooled beta to % change in relative protein expression, same
convention as the per-test cell: % change = (2^(-β) - 1) * 100
(positive = increase with WORSE cognition). z_dir is oriented higher = better,
so the sign convention matches the per-test rows. Appends COMBINED rows to
all_results_raw.csv for the forest / publication cells.
"""

combined_df = pd.read_csv(os.path.join(output_dir, 'combined_results_raw.csv'))

combined_df['pct_change']  = (2**(-combined_df['estimate']) - 1) * 100
combined_df['pct_ci_low']  = (2**(-combined_df['ci_high']) - 1) * 100
combined_df['pct_ci_high'] = (2**(-combined_df['ci_low'])  - 1) * 100
combined_df.to_csv(os.path.join(output_dir, 'combined_results_raw.csv'), index=False)

print("Combined cognition - percentage change in relative protein expression")
print("=" * 75)
print(combined_df[['biomarker', 'test', 'estimate',
                   'pct_change', 'p_value']].to_string(index=False))

all_results = pd.read_csv(os.path.join(output_dir, 'all_results_raw.csv'))
shared_cols = [c for c in all_results.columns if c in combined_df.columns]
all_results = all_results[all_results['test'] != 'COMBINED']  # avoid dupes on re-run
all_results = pd.concat([all_results, combined_df[shared_cols]], ignore_index=True)
all_results.to_csv(os.path.join(output_dir, 'all_results_raw.csv'), index=False)
print("\n\u2713 COMBINED rows appended to all_results_raw.csv "
      "(re-run forest / publication cells to include them)")